# Run AI Models on Colab / Kaggle → Access Locally via ngrok

This notebook turns a free Colab or Kaggle GPU runtime into a personal AI API server  
that you can query from any weak local device using simple HTTP requests.

**Flow:**
```
[Your Weak Device]  →  HTTP request  →  [ngrok tunnel]  →  [Colab/Kaggle GPU]
                    ←  JSON response ←                  ←  (model inference)
```

---
### Sections
1. **Configuration** — tokens & settings (fill this in first)
2. **Install dependencies**
3. **GPU info check**
4. **Load model**
5. **Define FastAPI server**
6. **Start ngrok tunnel & run server**
7. **Local client examples** — copy & run these on your own machine

## 1 — Configuration
Fill in your tokens below before running anything else.

In [ ]:
# ============================================================
#  FILL IN YOUR TOKENS / KEYS HERE
# ============================================================

# Get a free ngrok account and copy your authtoken from:
#   https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"

# (Optional) HuggingFace token — required for gated models
#   (Llama, Mistral, Gemma, etc.)  https://huggingface.co/settings/tokens
HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN"   # leave as-is if not needed

# Auto-generated every run — do NOT set this manually.
# A fresh 64-char hex secret is created here, embedded in the server,
# and exported to config.json so the website always has the matching key.
import secrets
API_SECRET_KEY = secrets.token_hex(32)
print(f"API_SECRET_KEY auto-generated ✓")

# ============================================================
#  MODEL SETTINGS
# ============================================================

# ✅ Chosen model: TinyLlama-1.1B-Chat — free, no token needed,
#    fits in ~2.2 GB VRAM, works on any Colab/Kaggle free tier GPU.
#
# Other drop-in options (just change MODEL_ID + TASK):
#   "microsoft/Phi-3-mini-4k-instruct"    (3.8B, stronger, ~USE_4BIT=False on T4)
#   "mistralai/Mistral-7B-Instruct-v0.3"  (7B, best quality,  USE_4BIT=True)
#   "HuggingFaceH4/zephyr-7b-beta"        (7B, great for chat, USE_4BIT=True)
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Task for the HuggingFace pipeline
TASK = "text-generation"

# Persona shown to the model as the system prompt (editable)
SYSTEM_PROMPT = "You are a helpful and friendly AI assistant. Keep answers concise."

# Use 4-bit quantization — set True for 7B+ models to fit in Colab VRAM
USE_4BIT = False

# Port the FastAPI server will listen on inside the runtime
SERVER_PORT = 8000

## 2 — Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn pyngrok transformers accelerate

# Install bitsandbytes only if 4-bit quantization is requested
import subprocess, sys
if USE_4BIT:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"])

print("Dependencies installed.")

## 3 — GPU Info Check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram = gpu.total_memory / 1024**3
    print(f"GPU  : {gpu.name}")
    print(f"VRAM : {vram:.1f} GB")
    print(f"CUDA : {torch.version.cuda}")
    DEVICE = 0
else:
    print("No GPU detected — running on CPU (inference will be slow).")
    DEVICE = -1

## 4 — Load Model

In [ ]:
from transformers import pipeline, BitsAndBytesConfig, AutoTokenizer
import os

# Log in to HuggingFace if a token was provided
hf_token = HF_TOKEN if HF_TOKEN != "YOUR_HUGGINGFACE_TOKEN" else None
if hf_token:
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to HuggingFace Hub.")

# Build quantization config if requested
quant_config = None
if USE_4BIT:
    quant_config = BitsAndBytesConfig(load_in_4bit=True)
    print("Using 4-bit quantization.")

# Load the tokenizer — needed to apply the model's chat template
print(f"Loading tokenizer for '{MODEL_ID}' ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)

# Load the model pipeline
print(f"Loading model '{MODEL_ID}' ...")
model_pipeline = pipeline(
    task=TASK,
    model=MODEL_ID,
    tokenizer=tokenizer,
    device=DEVICE if not USE_4BIT else None,
    model_kwargs={"quantization_config": quant_config} if quant_config else {},
    token=hf_token,
)
print("Model and tokenizer loaded successfully.")

## 5 — Define FastAPI Server

Four endpoints are exposed:

| Method | Path | Description |
|--------|------|-------------|
| GET | `/` | Health check + model info |
| POST | `/chat` | **Multi-turn chat** — send a `messages` array, get the assistant reply back |
| POST | `/predict` | Single raw-text inference (useful for non-chat models) |
| POST | `/predict_batch` | Batch raw-text inference |

The website uses **`/chat`**.  
`/predict` and `/predict_batch` are kept for experimenting with other pipeline tasks.

In [ ]:
from fastapi import FastAPI, HTTPException, Header
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional
import nest_asyncio

nest_asyncio.apply()

app = FastAPI(title="Colab/Kaggle AI Model Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


def check_auth(x_api_key: str = Header(default=None)):
    if API_SECRET_KEY == "change-me-to-something-random":
        return  # no auth configured
    if x_api_key != API_SECRET_KEY:
        raise HTTPException(status_code=401, detail="Invalid or missing x-api-key header")


# ── Schema ────────────────────────────────────────────────────

class Message(BaseModel):
    role: str       # "user" | "assistant" | "system"
    content: str

class ChatRequest(BaseModel):
    messages: List[Message]
    system_prompt: Optional[str] = None   # overrides the default SYSTEM_PROMPT
    max_new_tokens: int = 512
    temperature: float = 0.7
    top_p: float = 0.95

class SingleRequest(BaseModel):
    input: str
    kwargs: dict = {}

class BatchRequest(BaseModel):
    inputs: List[str]
    kwargs: dict = {}


# ── Endpoints ─────────────────────────────────────────────────

@app.get("/")
async def root():
    return {
        "status": "running",
        "model": MODEL_ID,
        "task": TASK,
        "device": "GPU" if DEVICE == 0 else "CPU",
    }


@app.post("/chat")
async def chat(req: ChatRequest, x_api_key: str = Header(default=None)):
    """
    Multi-turn chat endpoint.

    The client sends the full conversation history as a list of messages.
    The server applies the model's chat template, runs inference, and returns
    only the assistant's new reply.

    Request body:
      {
        "messages": [
          {"role": "user", "content": "Hello!"}
        ],
        "max_new_tokens": 512,
        "temperature": 0.7
      }
    """
    check_auth(x_api_key)

    # Build the full messages list with a system prompt at the front
    system_msg = req.system_prompt or SYSTEM_PROMPT
    full_messages = [{"role": "system", "content": system_msg}] + [
        {"role": m.role, "content": m.content} for m in req.messages
    ]

    # Apply the model's chat template to get a formatted prompt string
    prompt = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=True,  # adds the <|assistant|> token at the end
    )

    outputs = model_pipeline(
        prompt,
        max_new_tokens=req.max_new_tokens,
        temperature=req.temperature,
        top_p=req.top_p,
        do_sample=True,
        return_full_text=False,  # return only newly generated tokens
    )

    reply = outputs[0]["generated_text"].strip()
    return {"role": "assistant", "content": reply}


@app.post("/predict")
async def predict(req: SingleRequest, x_api_key: str = Header(default=None)):
    check_auth(x_api_key)
    if not req.input.strip():
        raise HTTPException(status_code=400, detail="'input' must not be empty")
    try:
        result = model_pipeline(req.input, **req.kwargs)
        return {"input": req.input, "output": result}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/predict_batch")
async def predict_batch(req: BatchRequest, x_api_key: str = Header(default=None)):
    check_auth(x_api_key)
    if not req.inputs:
        raise HTTPException(status_code=400, detail="'inputs' list must not be empty")
    try:
        results = model_pipeline(req.inputs, **req.kwargs)
        return {
            "outputs": [
                {"input": text, "output": out}
                for text, out in zip(req.inputs, results)
            ]
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print("FastAPI app defined — endpoints: GET /  POST /chat  POST /predict  POST /predict_batch")

## 6 — Start ngrok Tunnel & Run Server

This is the **last cell to run**. It will:
1. Connect ngrok and get the public URL
2. Auto-generate and download `config.json`
3. Start the server — **this cell blocks intentionally** (the server runs until you stop the notebook)

> **Why blocking instead of a background thread?**  
> Running uvicorn in a thread causes silent crashes in Colab/Kaggle due to event loop conflicts.  
> Running it directly with `nest_asyncio` is the stable approach — it simply means this cell never "finishes".

In [ ]:
import uvicorn, json, os, time, subprocess
import nest_asyncio
from pyngrok import ngrok

if not NGROK_AUTH_TOKEN or NGROK_AUTH_TOKEN == "YOUR_NGROK_AUTH_TOKEN":
    raise ValueError("Please set NGROK_AUTH_TOKEN in Cell 1 before running this cell.")

nest_asyncio.apply()

# ── Kill ALL existing ngrok processes at the OS level ────────
# ngrok.kill() only stops the local pyngrok process, but ngrok's
# servers keep the endpoint "online" until the TCP connection drops.
# pkill -f ngrok forces every ngrok binary to exit immediately,
# which triggers a clean disconnect on ngrok's side.
print("Cleaning up any existing ngrok sessions...")
subprocess.run(["pkill", "-9", "-f", "ngrok"], capture_output=True)
time.sleep(5)   # give ngrok's servers time to detect the disconnect

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Retry loop — ERR_NGROK_334 can still appear for a few seconds
# after the old process dies; we retry up to 3 times with a delay.
MAX_RETRIES = 3
for attempt in range(1, MAX_RETRIES + 1):
    try:
        tunnel    = ngrok.connect(SERVER_PORT)
        NGROK_URL = tunnel.public_url
        break
    except Exception as e:
        if attempt == MAX_RETRIES:
            raise RuntimeError(
                f"Could not start ngrok tunnel after {MAX_RETRIES} attempts.\n"
                "Go to https://dashboard.ngrok.com/tunnels and manually stop "
                "any active tunnels, then re-run this cell."
            ) from e
        print(f"Attempt {attempt} failed ({e.__class__.__name__}), retrying in 10 s...")
        time.sleep(10)

print("=" * 60)
print(f"  Public URL : {NGROK_URL}")
print("=" * 60)

# ── Export config.json and download it before the server blocks ──
config = {
    "ngrok_url":        NGROK_URL,
    "api_secret_key":   API_SECRET_KEY,
    "hf_token":         HF_TOKEN,
    "ngrok_auth_token": NGROK_AUTH_TOKEN,
    "model_id":         MODEL_ID,
    "task":             TASK,
    "system_prompt":    SYSTEM_PROMPT,
    "device":           "GPU" if DEVICE == 0 else "CPU",
    "use_4bit":         USE_4BIT,
    "server_port":      SERVER_PORT,
}
import base64
from IPython.display import display, HTML

json_str = json.dumps(config, indent=2)
with open("config.json", "w") as f:
    f.write(json_str)

# Non-blocking download link — works on Colab, Kaggle, and any Jupyter environment
b64 = base64.b64encode(json_str.encode()).decode()
display(HTML(f"""
<div style="margin:8px 0;padding:10px 16px;background:#1e3a2f;border-left:4px solid #4caf89;
            border-radius:6px;font-family:monospace;font-size:14px;color:#e2e4ef">
  ✅ config.json ready &nbsp;|&nbsp;
  <a href="data:application/json;base64,{b64}" download="config.json"
     style="color:#7c6af7;font-weight:bold;text-decoration:none">
     ⬇ Click here to download config.json
  </a>
  &nbsp;→ then place it in <code>local_client/</code> and refresh the website.
</div>
"""))
print(f"File also saved at: {os.path.abspath('config.json')}")

print("\nServer starting — this cell will stay running until you stop it.")
print("Stopping the cell or closing the notebook kills the server.\n")

# ── Run uvicorn via the event loop (avoids nest_asyncio + loop_factory conflict) ──
import asyncio
config = uvicorn.Config(app, host="0.0.0.0", port=SERVER_PORT, log_level="warning")
server = uvicorn.Server(config)
asyncio.get_event_loop().run_until_complete(server.serve())

## 7 — Notes

Cell 6 handles everything: ngrok tunnel, config export, and the server.  
Once Cell 6 is running you are done — no more cells to execute.

**Local client workflow (on your weak device):**
```
1. Drop config.json into  local_client/
2. python3 local_client/serve.py
3. Open  http://localhost:8080  →  auto-connects, start chatting
```

In [ ]:
# Config export and server startup are now combined in Cell 6.
# This cell is intentionally left empty — you can use it for quick tests
# while the server is running, e.g. a local pipeline call to verify the model:

# result = model_pipeline("Hello, how are you?", max_new_tokens=50)
# print(result[0]["generated_text"])

## Tips & Troubleshooting

### Every-session workflow
```
1. Run cells 1–5  →  model loads, FastAPI app defined
2. Run cell 6     →  ngrok connects, config.json downloaded, server starts (cell stays running)
3. Drop config.json into  local_client/
4. python3 local_client/serve.py
5. Open  http://localhost:8080  →  connects automatically
```

### Reference

| Topic | Notes |
|---|---|
| **Colab idle timeout** | Free tier disconnects after ~90 min idle. Use Kaggle (30 h/week) for longer sessions. |
| **Kaggle sessions** | Enable **Internet** in notebook Settings before running. |
| **ngrok free tier** | ~40 req/min, 1 active tunnel. URL changes each session unless you have a paid static domain. |
| **Large models** | Set `USE_4BIT = True` for 7B+ models. Colab T4 = 15 GB VRAM, Kaggle T4×2 = 30 GB. |
| **Security** | `API_SECRET_KEY` in `config.json` is sent with every request. Use a strong random value. |
| **Regenerate tokens** | If tokens are ever exposed, regenerate at ngrok.com and huggingface.co immediately. |